# Fisher Information Matrix

## Introduction

The **Fisher Information Matrix** quantifies the amount of information that observable data carries about unknown parameters. In logistic regression, it provides the theoretical foundation for:

1. **Asymptotic variance** of parameter estimates
2. **Standard errors** and confidence intervals
3. **Wald tests** for hypothesis testing
4. **Cramér-Rao lower bound** (efficiency of estimators)

---

## Definition

For a parameter vector $\theta \in \mathbb{R}^n$, the **Fisher Information Matrix** $I(\theta) \in \mathbb{R}^{n \times n}$ is defined as:

$$I(\theta) = \mathbb{E}\left[\left(\frac{\partial \log p(Y \mid X; \theta)}{\partial \theta}\right) \left(\frac{\partial \log p(Y \mid X; \theta)}{\partial \theta}\right)^T\right]$$

**Equivalently (under regularity conditions):**

$$I(\theta) = -\mathbb{E}\left[\frac{\partial^2 \log p(Y \mid X; \theta)}{\partial \theta \partial \theta^T}\right]$$

**Interpretation:** Expected curvature of the log-likelihood surface.

---

## Fisher Information for Logistic Regression

### Single Observation

For a single sample $(x, y)$, the log-likelihood contribution is:

$$\ell(\theta; x, y) = y \log p + (1-y) \log(1-p)$$

where $p = \sigma(\theta^T x)$.

**Gradient (score):**

$$\frac{\partial \ell}{\partial \theta} = (y - p) x$$

**Hessian:**

$$\frac{\partial^2 \ell}{\partial \theta \partial \theta^T} = -p(1-p) x x^T$$

**Fisher Information (single sample):**

$$I_1(\theta) = -\mathbb{E}\left[\frac{\partial^2 \ell}{\partial \theta \partial \theta^T}\right]$$

$$= \mathbb{E}[p(1-p) x x^T]$$

Since $p = \sigma(\theta^T x)$ is deterministic given $x$:

$$I_1(\theta) = p(1-p) x x^T$$

### Full Dataset

For $m$ independent samples $(x_1, y_1), \ldots, (x_m, y_m)$:

$$I(\theta) = \sum_{i=1}^{m} I_i(\theta) = \sum_{i=1}^{m} p_i(1-p_i) x_i x_i^T$$

**Matrix form:**

$$I(\theta) = X^T R X$$

where $R = \text{diag}(r_1, \ldots, r_m)$ with $r_i = p_i(1-p_i)$.

**Comparison with Hessian:**

From the Hessian derivation:

$$H = \frac{1}{m} X^T R X$$

**Therefore:**

$$\boxed{I(\theta) = m \cdot H}$$

**Key Insight:** The observed Hessian is proportional to the Fisher Information.

---

## Asymptotic Properties of MLE

Under regularity conditions, the **Maximum Likelihood Estimator (MLE)** $\hat{\theta}$ is:

1. **Consistent:** $\hat{\theta} \xrightarrow{p} \theta^*$ as $m \to \infty$
2. **Asymptotically normal:**

$$\sqrt{m}(\hat{\theta} - \theta^*) \xrightarrow{d} \mathcal{N}(0, I(\theta^*)^{-1})$$

3. **Asymptotically efficient:** Achieves the Cramér-Rao lower bound

**Practical implication:**

$$\hat{\theta} \sim \mathcal{N}\left(\theta^*, \frac{1}{m} I(\theta^*)^{-1}\right)$$

for large $m$.

---

## Standard Errors and Confidence Intervals

The **covariance matrix** of $\hat{\theta}$ is approximated by:

$$\text{Cov}(\hat{\theta}) \approx I(\hat{\theta})^{-1} = H^{-1} / m$$

**Standard error of $\hat{\theta}_j$:**

$$\text{SE}(\hat{\theta}_j) = \sqrt{[I(\hat{\theta})^{-1}]_{jj}}$$

**95% Confidence Interval:**

$$\hat{\theta}_j \pm 1.96 \cdot \text{SE}(\hat{\theta}_j)$$

**Interpretation:** With 95% confidence, the true parameter $\theta_j^*$ lies in this interval.

---

## Wald Test for Coefficient Significance

**Null hypothesis:** $H_0: \theta_j = 0$ (feature $j$ has no effect)

**Test statistic:**

$$z_j = \frac{\hat{\theta}_j}{\text{SE}(\hat{\theta}_j)}$$

**Under $H_0$:**

$$z_j \sim \mathcal{N}(0, 1)$$

**P-value (two-tailed):**

$$p = 2 \cdot P(|Z| > |z_j|) = 2 \cdot \Phi(-|z_j|)$$

where $\Phi$ is the standard normal CDF.

**Decision rule:**
- If $p < 0.05$: Reject $H_0$ → Feature is significant
- If $p \geq 0.05$: Fail to reject $H_0$ → Feature may not be informative

---

## Likelihood Ratio Test

**Alternative to Wald test:** Compare nested models.

**Full model:** $\mathcal{L}(\hat{\theta})$ with all features

**Restricted model:** $\mathcal{L}(\hat{\theta}_0)$ with feature $j$ removed

**Test statistic:**

$$\Lambda = 2[\ell(\hat{\theta}) - \ell(\hat{\theta}_0)]$$

**Under $H_0$:**

$$\Lambda \sim \chi^2(1)$$

**P-value:**

$$p = P(\chi^2(1) > \Lambda)$$

---

## Cramér-Rao Lower Bound

**Theorem:** For any unbiased estimator $\tilde{\theta}$:

$$\text{Cov}(\tilde{\theta}) \geq I(\theta)^{-1}$$

(in the sense of positive semi-definite ordering)

**Implication:** The MLE achieves this bound asymptotically → **Most efficient estimator**.

---

## Practical Computation

**Step 1:** Compute predicted probabilities $p_i = \sigma(\theta^T x_i)$

**Step 2:** Construct weight matrix $R = \text{diag}(p_1(1-p_1), \ldots, p_m(1-p_m))$

**Step 3:** Compute Fisher Information: $I(\hat{\theta}) = X^T R X$

**Step 4:** Invert to get covariance: $\text{Cov}(\hat{\theta}) = I(\hat{\theta})^{-1}$

**Step 5:** Extract standard errors: $\text{SE}(\hat{\theta}_j) = \sqrt{[\text{Cov}(\hat{\theta})]_{jj}}$

**Step 6:** Compute Wald statistics: $z_j = \hat{\theta}_j / \text{SE}(\hat{\theta}_j)$

**Step 7:** Compute p-values: $p_j = 2 \Phi(-|z_j|)$

---

## Numerical Stability

**Warning:** $I(\hat{\theta})$ may be near-singular if:
- Features are highly correlated (multicollinearity)
- Some $p_i \approx 0$ or $p_i \approx 1$ (perfect separation)

**Solutions:**
1. **Ridge regularization:** $I_{\text{reg}} = I(\hat{\theta}) + \lambda I$
2. **Remove redundant features**
3. **Use pseudo-inverse** if inversion fails

---

## Summary

| Concept | Formula |
|---------|---------|
| **Fisher Information** | $I(\theta) = X^T R X$ |
| **Relation to Hessian** | $I(\theta) = m \cdot H$ |
| **Covariance Matrix** | $\text{Cov}(\hat{\theta}) = I(\hat{\theta})^{-1}$ |
| **Standard Error** | $\text{SE}(\hat{\theta}_j) = \sqrt{[I(\hat{\theta})^{-1}]_{jj}}$ |
| **Wald Statistic** | $z_j = \hat{\theta}_j / \text{SE}(\hat{\theta}_j)$ |
| **Asymptotic Distribution** | $\hat{\theta} \sim \mathcal{N}(\theta^*, I(\theta^*)^{-1})$ |

**Key Takeaway:** The Fisher Information Matrix is the bridge between optimization (Hessian) and statistical inference (standard errors, p-values, confidence intervals).
